# black-box regression results: improvability

Mean RMSE-based improvability (%) versus the number of sampled expressions, and small heatmap tables for the ablation comparisons.

In [ ]:
import pandas as pd
import json
import os
import numpy as np; np.random.seed(42)
from glob import glob
from tqdm import tqdm
import seaborn as sns
import pdb

import sys; sys.path.append('..')

import matplotlib.pyplot as plt
import matplotlib

matplotlib.rc('pdf', fonttype=42)
matplotlib.rc('ps', fonttype=42)

params = {'mathtext.default': 'regular' }          
plt.rcParams.update(params)

sns.set(font_scale=1.25, style='whitegrid')
sns.set(palette='magma')
sns.set_style("whitegrid")

if True: # Off-the-shelf evaluation
    rdir  = '../../results/black-box/'
    figdir = f'../figs/black-box/'
else: # with HP tuning
    rdir  = '../../results/black-box-tuned/'
    figdir = f'../figs/black-box-tuned/'

datadir = '../../datasets/blackbox/'

pretty_labels = {'model_size': 'Mean Expression Size', 'r2_test_rank': 'Mean Test $R^2$ Rank', 'r2_test_norm': 'Mean Normalized $R^2$'}

def save(name='tmp',h=None):
    name = name.strip().replace(' ','-').replace('%','pct')
    if h == None:
        h = plt.gcf()

    h.tight_layout()

    print('saving',figdir+'/'+name+'.pdf')

    if not os.path.exists(figdir):
        os.makedirs(figdir)

    plt.savefig(figdir+'/'+name+'.pdf', bbox_inches='tight')


# read data from feather

In [ ]:
df_results = pd.read_feather(rdir+'results.feather')
df_sum = pd.read_csv(rdir+'results-summary.csv.gz', compression='gzip')

df_results = df_results[~df_results['algorithm'].isin(['Genetic Engine rs', 'Genetic Engine hc', 'Genetic Engine 1p1'])]
df_sum = df_sum[~df_sum['algorithm'].isin(['Genetic Engine rs', 'Genetic Engine hc', 'Genetic Engine 1p1'])]

df_results = df_results.replace({'Genetic Engine': 'G. Engine'})
df_sum = df_sum.replace({'Genetic Engine': 'G. Engine'})

print('mean trial count:')
print(df_results.groupby('algorithm')['dataset'].count().sort_values()
      / df_results.dataset.nunique())

In [ ]:
from scripts.impute_missing_runs import impute_missing_runs

_impute_missing_runs = True
_impute_r2_source     = 'fixed'  # 'fixed' or 'linear'
_impute_r2_test       = 0.0
_impute_model_size    = "dataset_median"  # 'dataset_median', 'algorithm_median', 'algorithm_mean', 'linear_expression_size', or a fixed number
_impute_algorithms    = None
_linear_algorithm_name = 'Linear'

if _impute_missing_runs:
    df_results, df_sum = impute_missing_runs(
        df_results, df_sum,
        datadir=datadir,
        impute_r2_source=_impute_r2_source,
        impute_r2_test=_impute_r2_test,
        impute_model_size=_impute_model_size,
        impute_algorithms=_impute_algorithms,
        linear_algorithm_name=_linear_algorithm_name,
    )


In [ ]:
_r2_norm_strategy = 'median'  # 'minmax': (method - worst) / (best - worst)
                               # 'median': (method - median) / (best - median)
_r2_norm_clip_negative = True  # clip normalized scores below 0 to 0 (keeps the scale in [0, 1] with the median strategy)
_R2_NORM_MIN_DENOM = 1e-5
_IMPROVABILITY_MIN_ERR = 1e-12

def _normalize_r2(x, strategy, clip_negative=None):
    clip_negative = _r2_norm_clip_negative if clip_negative is None else clip_negative
    best = x.max()
    baseline = x.min() if strategy == 'minmax' else x.median()
    denom = best - baseline
    if denom == 0:
        return pd.Series(1.0, index=x.index)
    norm = (x - baseline) / max(denom, _R2_NORM_MIN_DENOM)
    return norm.clip(lower=0) if clip_negative else norm

def _improvability(err):
    """Relative headroom to the best method in percent, (err - best_err) / err * 100 (lower is better)."""
    best = err.min()
    return ((err - best) / err.clip(lower=_IMPROVABILITY_MIN_ERR) * 100).clip(lower=0, upper=100)

def _apply_excluding_similar(group, col, fn, decimals=3, cohort=None):
    """Apply `fn` to `col` within a dataset, with reference stats from the non-rasr methods plus each rasr cohort."""
    is_special = group['*algorithm*'].str.contains('rasr', case=False)
    out = pd.Series(index=group.index, dtype=float)
    base = group.loc[~is_special, col]
    out.loc[~is_special] = fn(base)

    special_idx = group.index[is_special]
    labels = pd.Series(special_idx, index=special_idx, dtype=object)
    if cohort is not None:
        given = cohort.reindex(special_idx)
        labels.loc[given.notna()] = given.loc[given.notna()]

    for _, idxs in labels.groupby(labels, sort=False).groups.items():
        idxs = list(idxs)
        combined = pd.concat([base, group.loc[idxs, col]])
        scored = fn(combined)
        for idx in idxs:
            out.loc[idx] = round(scored.loc[idx], decimals)
    return out

def r2_norm_excluding_similar(group, r2_col='r2_test', strategy=None, clip_negative=None, cohort=None):
    strategy = strategy or _r2_norm_strategy
    return _apply_excluding_similar(
        group, r2_col, lambda x: _normalize_r2(x, strategy, clip_negative), cohort=cohort)

def improvability_excluding_similar(group, err_col='rmse_test', cohort=None):
    return _apply_excluding_similar(group, err_col, _improvability, cohort=cohort)


In [ ]:
def compare(df_compare, x='r2_test',y='algorithm', row=None, col=None, scale=None, xlim=[],**kwargs):
    plt.figure()
    order = df_compare.groupby(y)[x].median().sort_values(ascending=False).index
    if scale=='log' and len(xlim)>0 and xlim[0] == 0:
        df_compare.loc[:,x] += 1
        xlim[0] = 1
        xnew = '1 + '+x
        df_compare=df_compare.rename(columns={x:xnew})
        x = xnew
    
    sns.catplot(data=df_compare, 
                kind='point',
                y=y,
                x=x,
                order=order,
                fliersize=0,
                row=row,
                col=col,
                **kwargs
               )

    plt.ylabel('')
    if len(xlim)>0:
        plt.xlim(xlim[0],xlim[1])
    if scale:
        plt.gca().set_xscale(scale)
    
    save(name='_'.join(['cat-boxplot',x + '-by-'+ y]) )
    if col:
        save(name='_'.join(['cat-boxplot',x + '-by-'+ y] + [col]) )



# excluded_datasets = ['1089_USCrime', '1193_BNG_lowbwt', '1199_BNG_echoMonths', '650_fri_c0_500_50']
# excluded_datasets = ['1089_USCrime', '1193_BNG_lowbwt', '1199_BNG_echoMonths', '650_fri_c0_500_50', '678_visualizing_environmental', '210_cloud', '192_vineyard']
excluded_datasets = []

excluded_methods = ['Linear']  # 'Linear' is only an imputation source; dropped from all plots and normalizations

alg_name_map = df_sum[['algorithm', '*algorithm*']].drop_duplicates().set_index('algorithm')['*algorithm*']

df_sum_filtered = df_sum[~df_sum['dataset'].isin(excluded_datasets)].copy()

df_sum_filtered['*algorithm*'] = df_sum_filtered['*algorithm*'].str.lstrip('*')


df_sum_filtered = df_sum_filtered[~df_sum_filtered['*algorithm*'].isin(excluded_methods)]

print(f"Remaining: {df_sum_filtered['dataset'].nunique()} datasets")
print(df_sum_filtered['dataset'].unique())

In [ ]:
import re

# "best" for improvability is computed against one fixed pool: every plain baseline plus every rasr variant
# shown anywhere in this notebook

# the plain 94M post-trained runs ('rasr_94m' = 1024 candidates, '_c<N>', '_greedy'), the GRPO sweep and
# the Base Model sweep
_pt_series_filters = {
    'RS-GRPO':    r'^rasr_94m(?:_c\d+|_greedy)?$',
    'GRPO':       r'^rasr_94m_grpo(?:_c\d+|_greedy)$',
    'Base Model': r'^rasr_94m_base(?:_c\d+|_greedy)$',
}

def _n_candidates(name):
    if name == 'rasr_94m':
        return 1024
    if name.endswith('_greedy'):
        return 1
    m = re.search(r'_c(\d+)$', name)
    return int(m.group(1)) if m else None

_is_special_all = df_results['algorithm'].str.contains('rasr', case=False)
_pt_pattern = '|'.join(_pt_series_filters.values())
_pt_sweep_algorithms = set(
    df_results.loc[
        _is_special_all
        & df_results['algorithm'].apply(lambda a: _n_candidates(a) is not None)
        & df_results['algorithm'].str.contains(_pt_pattern, case=False),
        'algorithm'
    ]
)

_c1_algorithms = {
    ('Post-trained', '94M'): 'rasr_94m',
    ('Base',         '94M'): 'rasr_94m_base_c256',
    ('Post-trained', '24M'): 'rasr_24m',
    ('Base',         '24M'): 'rasr_24m_base',
}
_c2_algorithms = {
    ('Post-trained', 'New prior'): 'rasr_24m',
    ('Base',         'New prior'): 'rasr_24m_base',
    ('Post-trained', 'Old prior'): 'rasr_24m_simple_prior',
    ('Base',         'Old prior'): 'rasr_24m_simple_prior_base',
}
_c3_algorithms = {
    'Final':           'rasr_24m',
    'Skeleton':        'rasr_24m_skeleton',
    'No Refinement':   'rasr_24m_no_refine',
    'No Init Noise':   'rasr_24m_no_noise',
}

ALL_SHOWN_ALGORITHMS = (
    _pt_sweep_algorithms
    | set(_c1_algorithms.values())
    | set(_c2_algorithms.values())
    | set(_c3_algorithms.values())
)

_display_stripped = alg_name_map.str.lstrip('*')
_is_special_by_raw = alg_name_map.index.str.contains('rasr', case=False)
ALL_BASELINE_ALGORITHMS = set(_display_stripped.index[~_is_special_by_raw & ~_display_stripped.isin(excluded_methods)])

ALL_METHODS = ALL_BASELINE_ALGORITHMS | ALL_SHOWN_ALGORITHMS

print(f'{len(ALL_BASELINE_ALGORITHMS)} plain baselines + {len(ALL_SHOWN_ALGORITHMS)} rasr variants '
      f'= {len(ALL_METHODS)} methods considered as "best" candidates throughout this notebook:')
print(sorted(ALL_METHODS))


## Post-training Comparison

Mean improvability (%) vs. number of sampled candidates, for the RS-GRPO, GRPO, and Base Model series.

In [ ]:
# --- configure here ---
_seed_agg_extra = 'median'  # 'mean' or 'median' over per-seed trials per dataset+algorithm (datasets: always mean)
_series_filters = _pt_series_filters  # RS-GRPO / GRPO / Base Model substrings, defined in the algorithm-registry cell above
_include_greedy = False  # False drops the '_greedy' runs entirely (dashed reference line and their share of the cohort), as in the paper
_series_colors = {
    'RS-GRPO':    '#2a78d6',
    'GRPO':       '#008300',
    'Base Model': '#e87ba4',
}
_extra_metric = 'rmse_test_improvability'  # y-axis metric: 'r2_test_norm' (higher is better) or 'rmse_test_improvability' (lower is better)
_paper_style = True
_fig_width_in, _fig_height_in = 3.3, 2.5
# ----------------------

from matplotlib.lines import Line2D

df_extra = df_results.copy()
df_extra = df_extra[~df_extra['dataset'].isin(excluded_datasets)]
df_extra['*algorithm*'] = df_extra['algorithm'].map(alg_name_map).str.lstrip('*')
df_extra = df_extra[~df_extra['*algorithm*'].isin(excluded_methods)]

df_extra = df_extra[df_extra['algorithm'].isin(ALL_METHODS)]

agg_fn_extra = 'mean' if _seed_agg_extra == 'mean' else 'median'
df_extra_agg = (
    df_extra.groupby(['algorithm', '*algorithm*', 'dataset'])[['r2_test', 'mse_test']]
    .agg(agg_fn_extra)
    .reset_index()
)
df_extra_agg['rmse_test'] = np.sqrt(df_extra_agg['mse_test'])

# candidate count: bare 'rasr_94m' = 1024, '_c<N>' gives the count, '_greedy' = 1 candidate at temperature 0;
# other names are dropped
df_extra_agg['n_candidates'] = df_extra_agg['algorithm'].apply(_n_candidates)

if not _include_greedy:
    df_extra_agg = df_extra_agg[df_extra_agg['n_candidates'] != 1]

# cohort: all ALL_SHOWN_ALGORITHMS are scored together against the plain baselines and each other (against
# the baselines alone they would all score the best possible)
_cohort = pd.Series(np.where(df_extra_agg['algorithm'].isin(ALL_SHOWN_ALGORITHMS), 'shown', None), index=df_extra_agg.index)

_extra_metric_fn = {
    'r2_test_norm': r2_norm_excluding_similar,
    'rmse_test_improvability': improvability_excluding_similar,
}[_extra_metric]
df_extra_agg[_extra_metric] = df_extra_agg.groupby('dataset', group_keys=False).apply(
    lambda g: _extra_metric_fn(g, cohort=_cohort.reindex(g.index)))

df_extra_agg = df_extra_agg[df_extra_agg['n_candidates'].notna()]

_mean_col = 'mean_' + _extra_metric
summary_extra = (
    df_extra_agg
    .groupby(['algorithm', 'n_candidates'])[_extra_metric]
    .mean()
    .reset_index()
    .rename(columns={_extra_metric: _mean_col})
)

# paper styling scoped to this figure via rc_context (Times, thin lines, 8pt), so the heatmaps keep the sns theme
_paper_rc = {
    'font.family': 'serif',
    'font.serif': ['Nimbus Roman', 'Times New Roman', 'Liberation Serif', 'DejaVu Serif'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Nimbus Roman',
    'mathtext.it': 'Nimbus Roman:italic',
    'mathtext.bf': 'Nimbus Roman:bold',
    'mathtext.cal': 'Nimbus Roman:italic',
    'mathtext.default': 'regular',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'legend.frameon': False,
    'font.size': 8,
    'axes.labelsize': 8,
    'axes.titlesize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
} if _paper_style else {}

with plt.rc_context(_paper_rc):
    fig, ax = plt.subplots(figsize=(_fig_width_in, _fig_height_in) if _paper_style else (6, 5))

    _candidate_grid = [16, 64, 256, 1024]
    _any_greedy = False

    for series_name, name_filter in _series_filters.items():
        _mask = summary_extra['algorithm'].str.contains(name_filter, case=False)
        _series = summary_extra[_mask].sort_values('n_candidates')
        print(f'{series_name}: {list(_series["algorithm"])}')

        _sampled = _series[_series['n_candidates'].isin(_candidate_grid)]
        _greedy = _series[_series['n_candidates'] == 1]
        _color = _series_colors[series_name]

        ax.plot(
            _sampled['n_candidates'], _sampled[_mean_col], marker='o', label=series_name, color=_color,
            linewidth=1.3 if _paper_style else 1.5, markersize=3.5 if _paper_style else 6,
        )

        if not _greedy.empty:
            ax.axhline(_greedy[_mean_col].iloc[0], color=_color, linestyle='--', alpha=0.7, linewidth=0.9 if _paper_style else 1.5)
            _any_greedy = True

    ax.set_xscale('log', base=2)
    ax.set_xticks(_candidate_grid)
    ax.set_xticklabels([str(c) for c in _candidate_grid])
    _extra_ylabel_map = {'r2_test_norm': 'Mean Normalized $R^2$',
                         'rmse_test_improvability': 'Mean Improvability (%)'}
    ax.set_xlabel('Sampled Expressions')
    ax.set_ylabel(_extra_ylabel_map[_extra_metric])

    _handles, _labels = ax.get_legend_handles_labels()
    if _any_greedy:
        _handles.append(Line2D([0], [0], color='gray', linestyle='--', alpha=0.7))
        _labels.append('Greedy')

    if _paper_style:
        ax.legend(_handles, _labels, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=len(_handles), handlelength=1.5, columnspacing=1.0)
        sns.despine(ax=ax)
    else:
        ax.legend(_handles, _labels, loc='upper right')
        sns.despine(left=True, bottom=True)

    if _paper_style:
        # no bbox_inches='tight': it would resize the PDF and rescale the fonts
        fig.tight_layout(pad=0.3)
        os.makedirs(figdir, exist_ok=True)
        print('saving', figdir + '/post_training_sweep.pdf')
        fig.savefig(figdir + '/post_training_sweep.pdf')
    else:
        plt.tight_layout()
        save(name=f'post_training_sweep')


# Improvability comparison tables

Each heatmap shows mean RMSE-based improvability (%) -- `(err - best_err) / err * 100` per dataset, then averaged across datasets; **lower is better**, 0 = best method on that dataset. "Best" is always computed per dataset over `ALL_METHODS` (every plain baseline plus every rasr variant that appears in any plot/table in this notebook, defined in the algorithm-registry cell above) -- so the same fixed pool of methods is used as the reference everywhere, not a different subset per comparison.

In [ ]:
# --- configure here ---
_seed_agg = 'median'  # 'mean' or 'median' over per-seed trials of each dataset+algorithm, before computing improvability
# ----------------------

def mean_improvability(algorithms, seed_agg=_seed_agg):
    """Mean RMSE-based improvability (%) of `algorithms`, scored against the fixed ALL_METHODS pool with a shared rasr cohort."""
    data = df_results.copy()
    data = data[~data['dataset'].isin(excluded_datasets)]
    data['*algorithm*'] = data['algorithm'].map(alg_name_map).str.lstrip('*')
    data = data[~data['*algorithm*'].isin(excluded_methods)]

    data = data[data['algorithm'].isin(ALL_METHODS)]

    data['rmse_test'] = np.sqrt(data['mse_test'])
    agg_fn = 'mean' if seed_agg == 'mean' else 'median'
    data = data.groupby(['algorithm', '*algorithm*', 'dataset'])['rmse_test'].agg(agg_fn).reset_index()

    # cohort built after the groupby-agg (which resets the index), else it fails to align in reindex()
    _cohort = pd.Series(np.where(data['algorithm'].isin(ALL_SHOWN_ALGORITHMS), 'shown', None), index=data.index)
    data['rmse_test_improvability'] = data.groupby('dataset', group_keys=False).apply(
        lambda g: improvability_excluding_similar(g, err_col='rmse_test', cohort=_cohort.reindex(g.index)))

    return data.groupby('algorithm')['rmse_test_improvability'].mean().reindex(algorithms)


def plot_improvability_heatmap(table, title, name, figsize=None, cbar=True):
    """Render a table of improvability values (%) as an annotated heatmap (darker = better)."""
    if figsize is None:
        figsize = (0.9 * table.shape[1] + 2.2, 0.9 * table.shape[0] + 1.6)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(table, annot=True, fmt='.2f', cmap='viridis_r', linewidths=0.5, linecolor='white',
                cbar=cbar, cbar_kws={'label': 'Mean Improvability (%)'} if cbar else None, ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    save(name=name)


## Post-training × Model Size

`rasr_94m`, `rasr_94m_base_c256`, `rasr_24m`, `rasr_24m_base`

In [ ]:
_c1_values = mean_improvability(list(_c1_algorithms.values()))
_c1_table = pd.DataFrame(index=['Post-trained', 'Base'], columns=['94M', '24M'], dtype=float)
for (row, col), alg in _c1_algorithms.items():
    _c1_table.loc[row, col] = _c1_values[alg]

plot_improvability_heatmap(_c1_table, 'Post-training × Model Size', 'improvability_heatmap_pretraining_size')


## Prior Ablation

`rasr_24m`, `rasr_24m_base`, `rasr_24m_simple_prior`, `rasr_24m_simple_prior_base`

In [ ]:
_c2_values = mean_improvability(list(_c2_algorithms.values()))
_c2_table = pd.DataFrame(index=['Post-trained', 'Base'], columns=['New prior', 'Old prior'], dtype=float)
for (row, col), alg in _c2_algorithms.items():
    _c2_table.loc[row, col] = _c2_values[alg]

plot_improvability_heatmap(_c2_table, 'Prior Ablation', 'improvability_heatmap_prior_ablation')


## Constant Ablation

`rasr_24m`, `rasr_24m_skeleton`, `rasr_24m_no_refine`, `rasr_24m_no_noise`

In [ ]:
_c3_values = mean_improvability(list(_c3_algorithms.values()))
_c3_table = pd.DataFrame({'Improvability (%)': _c3_values.values}, index=list(_c3_algorithms.keys()))

plot_improvability_heatmap(_c3_table, 'Constant Ablation', 'improvability_heatmap_constant_ablation', cbar=False)
